# Семинар 16. Python. Работа с аудио и видео

## Мотивация

Модель не умеет слушать wav. Чтобы обучить распознавание речи, детектор поломки
станка по звуку или классификатор жанра, запись сначала превращают в **признаки**
— почти всегда в спектрограмму, то есть в картинку «частота × время». Весь путь от
файла до этой картинки лежит на вас, и на нём три места, где данные молча портятся.

1. **Частота дискретизации.** `librosa.load` по умолчанию приводит любой файл к
   22050 Гц и к моно. Половину записей это устраивает, а у второй половины
   бесследно исчезает всё выше 11 кГц — и модель учится не на том, что вы думали.
2. **Форма массива.** У `soundfile` стерео — это `(сэмплы, каналы)`, у `librosa` и
   `torchaudio` — `(каналы, сэмплы)`. Перепутали — и «длительность записи» стала
   равна 0.00009 секунды. Исключения при этом не будет.
3. **Одного сигнала во времени мало.** По графику амплитуды не видно, какие
   частоты звучали. Нужен переход в частотную область — и сразу выясняется, что
   спектр всей записи целиком тоже не годится, если звук меняется.

Сегодня проходим путь целиком: сэмплы и частота дискретизации → чтение и запись
wav → амплитуда и клиппинг → спектр → STFT и спектрограмма → шкала дБ и мел-шкала
→ те же операции тензорами в `torchaudio` → коротко видео как последовательность
кадров. Домашка — «спектрограмма аудиозаписи», так что получившийся конвейер
пригодится сразу.

Что уже знакомо и повторяться не будет: массивы `numpy` (семинар 14), графики
`matplotlib` (семинар 1), работа с изображениями (семинар 15). Спектрограмма и
кадр видео — ровно такие же двумерные и трёхмерные массивы, как картинка, и всё,
что вы умеете делать с изображением, применимо и к ним.

## 1. Звук — это массив чисел

Микрофон измеряет давление воздуха и записывает результат числами через равные
промежутки. Одно измерение — **сэмпл**, число измерений в секунду — **частота
дискретизации** (sample rate, `sr`, Гц). Отсюда главное соотношение семинара:

`длительность = число сэмплов / sr`

Сгенерируем чистый тон — синус частоты 440 Гц (нота «ля»).

In [ ]:
import numpy as np

sr = 22050
t = np.arange(sr * 2) / sr
y = 0.5 * np.sin(2 * np.pi * 440 * t)
print(y.shape, y.dtype)
print(len(y) / sr, "секунд")

`t` — моменты времени сэмплов с шагом `1/sr`, массив `y` — сам звук. Никакого
«аудиоформата» в памяти нет: это обычный массив `float64` со значениями от -1 до 1.

Посмотрим на первые 5 миллисекунд.

In [ ]:
import matplotlib.pyplot as plt

plt.plot(t[:110], y[:110])
plt.xlabel("время, с")
plt.ylabel("амплитуда")
plt.show()

#### ❓ **Вопрос**: Тот же звук сгенерировали с `sr = 44100`. Как изменятся форма массива и длительность?

<details>

<summary><strong>Ответ</strong></summary>

Форма станет `(88200,)` — чисел вдвое больше, потому что вдвое больше измерений в секунду. Длительность не изменится: она считается как `len(y) / sr`, и в дроби выросли и числитель, и знаменатель — `88200 / 44100 = 2.0`, как в ячейке выше было `44100 / 22050 = 2.0`.

На графике при этом на тех же 5 мс окажется вдвое больше точек: форма волны описана подробнее.

</details>

## 2. Запись и чтение wav: soundfile

Массив в файл и обратно проще всего гонять библиотекой `soundfile` (обёртка над
libsndfile). Формат `wav` — это небольшой заголовок и подряд идущие сэмплы; самый
ходовой вариант хранения — 16-битные целые, `PCM_16`.

In [ ]:
import soundfile as sf
from pathlib import Path

work = Path("/tmp/course-audio")
work.mkdir(exist_ok=True)
sf.write(work / "tone.wav", y, sr, subtype="PCM_16")
print((work / "tone.wav").stat().st_size, "байт")

44100 сэмплов по 2 байта плюс 44 байта заголовка — размер файла сходится
руками. Прочитаем обратно.

In [ ]:
data, sr_read = sf.read(work / "tone.wav")
print(data.shape, data.dtype, sr_read)
print(float(np.abs(data - y).max()))

`sf.read` возвращает **пару** «массив, частота дискретизации»: частота живёт в
файле, а не в массиве, и потерять её нельзя. Сэмплы по умолчанию приводятся к
`float64` в диапазоне -1..1, каким бы ни был формат внутри файла.

#### ❓ **Вопрос**: Прочитанный массив не совпал с исходным в точности. Откуда взялось расхождение порядка 3e-5?

<details>

<summary><strong>Ответ</strong></summary>

Мы записали файл как `PCM_16`: каждый сэмпл округлён до одного из 65536 уровней. Шаг сетки — `1/32768 ≈ 3.05e-5`, максимальная ошибка округления равна половине шага, и напечатанное `3.05e-5` — это она с учётом несимметричности диапазона. Такое округление называется **квантованием**, и оно необратимо: исходные `float64` в файле не сохранились.

Если потери недопустимы (например, вы сохраняете промежуточный результат обработки), пишите `subtype='FLOAT'` — тогда расхождение будет нулевым, но файл станет вдвое больше.

</details>

Метаданные можно узнать, не читая сами сэмплы, — это важно, когда файл на
несколько часов и в память целиком не влезает.

In [ ]:
info = sf.info(work / "tone.wav")
print(info.samplerate, info.channels, info.frames)
print(info.duration, info.subtype)

## 3. Моно и стерео: форма массива

**Канал** — отдельная дорожка звука (левая, правая). Соберём стереофайл: слева
440 Гц, справа 880 Гц.

In [ ]:
right = 0.25 * np.sin(2 * np.pi * 880 * t)
stereo = np.stack([y, right], axis=1)
sf.write(work / "stereo.wav", stereo, sr, subtype="PCM_16")
st, _ = sf.read(work / "stereo.wav")
print(st.shape)
print(sf.info(work / "stereo.wav").channels, sf.info(work / "stereo.wav").frames)

У `soundfile` форма стерео — `(сэмплы, каналы)`: канал идёт **последним**
измерением, а `frames` в метаданных — это число сэмплов на канал, а не общее число
чисел. Сведение в моно — усреднение по каналам.

In [ ]:
mono = st.mean(axis=1)
print(mono.shape)
print(round(float(np.abs(st[:, 0]).max()), 3),
      round(float(np.abs(st[:, 1]).max()), 3))

#### ❓ **Вопрос**: Скрипт считает длительность как `len(x) / sr`. Что он вернёт для нашего стереофайла и почему на это нельзя полагаться?

<details>

<summary><strong>Ответ</strong></summary>

Здесь повезёт: `len` двумерного массива — длина **первого** измерения, а у `soundfile` первым идёт число сэмплов, значит `44100 / 22050 = 2.0` — верно.

Но стоит прочитать тот же файл библиотекой, кладущей канал первым (это `librosa` с `mono=False` и `torchaudio` — увидим в разделах 4 и 10), как `len` станет равен 2, а длительность — `2 / 22050 ≈ 0.00009` секунды. Исключения не будет, будет тихо неверный результат. Смотреть надо на `.shape` целиком.

</details>

## 4. librosa.load: удобно и опасно

`librosa` — основная библиотека анализа звука в Python. Её `load` читает почти
любой формат, не только wav, но по дороге **меняет данные**. Запишем тон на
«музыкальной» частоте 44100 Гц и прочитаем без аргументов.

In [ ]:
import librosa

t44 = np.arange(44100 * 2) / 44100
y44 = 0.5 * np.sin(2 * np.pi * 440 * t44)
sf.write(work / "tone44k.wav", y44, 44100, subtype="PCM_16")
loaded, sr_loaded = librosa.load(work / "tone44k.wav")
print(sr_loaded, loaded.shape, loaded.dtype)

В файле 88200 сэмплов при 44100 Гц, а получили 44100 при 22050 Гц. Это
умолчание `sr=22050`: `librosa.load` **ресемплирует** запись. Отключается явным
`sr=None` — «оставить как в файле».

In [ ]:
native, sr_native = librosa.load(work / "tone44k.wav", sr=None)
print(sr_native, native.shape)
print(librosa.get_duration(y=loaded, sr=sr_loaded),
      librosa.get_duration(y=native, sr=sr_native))

#### ❓ **Вопрос**: Массив после `librosa.load` вдвое короче, а `get_duration` в обоих случаях даёт 2.0. Что тогда всё-таки потерялось?

<details>

<summary><strong>Ответ</strong></summary>

Длительность и не могла потеряться: сократились и число сэмплов, и частота дискретизации, дробь `len(y)/sr` осталась прежней. Потерялись **высокие частоты**. При `sr = 22050` представимы частоты только до `sr/2 = 11025` Гц (предел Найквиста), всё выше при ресемплинге отфильтровывается.

Нашему тону 440 Гц это безразлично, а записи с шипящими согласными, тарелками или ультразвуковым датчиком — нет. Поэтому в исследовательском коде пишут `sr=None` или явное нужное значение, а не полагаются на умолчание.

</details>

Второе умолчание — `mono=True`: любые каналы схлопываются в один. Проверим на
нашем стереофайле.

In [ ]:
one, _ = librosa.load(work / "stereo.wav", sr=None)
both, _ = librosa.load(work / "stereo.wav", sr=None, mono=False)
print(one.shape, both.shape)

#### ❓ **Вопрос**: `soundfile` вернул для того же файла `(44100, 2)`, а `librosa` с `mono=False` — `(2, 44100)`. Кто из них ошибается?

<details>

<summary><strong>Ответ</strong></summary>

Никто: это разные соглашения. `soundfile` следует раскладке внутри файла, где сэмплы каналов чередуются, и кладёт канал последним измерением. `librosa` (и, как увидим, `torchaudio`) кладёт канал первым, потому что дальше по конвейеру удобно обрабатывать каждый канал как отдельную строку матрицы.

Практический вывод один: после любой загрузки печатайте `.shape` и сверяйтесь с ожидаемой длительностью — как в вопросе про `len(x) / sr` выше.

</details>

## 5. Амплитуда, нормализация, клиппинг

Диапазон -1..1 — не украшение: за его пределами сэмпл в файл просто не запишется.
Записи приходят разной громкости, и перед подачей в модель их приводят к одному
масштабу. Самый простой способ — **пик-нормализация**: поделить на максимум
модуля.

In [ ]:
quiet = 0.05 * np.sin(2 * np.pi * 220 * t)
print(round(float(np.abs(quiet).max()), 4))
norm = quiet / np.abs(quiet).max()
print(round(float(np.abs(norm).max()), 4))

Соблазн «сделать погромче» умножением на константу заканчивается
**клиппингом**: всё, что вышло за диапазон, срезается по границе.

In [ ]:
loud = np.clip(quiet * 30, -1.0, 1.0)
print(int(np.sum(np.abs(quiet * 30) > 1.0)), "сэмплов из", len(quiet))
print(round(float(np.abs(quiet * 30).max()), 3),
      round(float(np.abs(loud).max()), 3))

Посмотрим, во что превратился спектр. Забегая в раздел 6: `np.fft.rfft`
показывает, из каких частот собран сигнал.

In [ ]:
spec_loud = np.abs(np.fft.rfft(loud))
grid = np.fft.rfftfreq(len(loud), 1 / sr)
print(sorted(float(grid[i]) for i in np.argsort(spec_loud)[-4:]))

#### ❓ **Вопрос**: Мы срезали верхушки у 23600 сэмплов из 44100. Синус же остался синусом — почему это порча данных?

<details>

<summary><strong>Ответ</strong></summary>

Он как раз перестал быть синусом: у волны появились плоские площадки на уровне ±1, то есть это уже другой сигнал. Последняя ячейка это и показывает: кроме исходных 220 Гц, в спектре появились 660, 1100 и 1540 Гц — нечётные гармоники, которых в `quiet` не было. На слух это хрип.

Поэтому громкость правят делением на пик (или нормировкой по RMS), а не умножением наугад: `norm` из предыдущей ячейки громче ровно в 20 раз и при этом не искажён.

</details>

## 6. Одного времени мало: спектр

Сложим два тона — 440 и 660 Гц. Это **аккорд**: на графике во времени видна волна
сложной формы, и по ней не сказать, из чего она собрана.

In [ ]:
chord = 0.6 * np.sin(2 * np.pi * 440 * t) + 0.3 * np.sin(2 * np.pi * 660 * t)
plt.plot(t[:220], chord[:220])
plt.xlabel("время, с")
plt.ylabel("амплитуда")
plt.show()

Преобразование Фурье раскладывает сигнал по синусам разных частот. Для
вещественного сигнала берут `np.fft.rfft` (вторая половина спектра симметрична и не
нужна), а сетку частот к нему даёт `np.fft.rfftfreq`.

In [ ]:
spectrum = np.abs(np.fft.rfft(chord))
freqs = np.fft.rfftfreq(len(chord), 1 / sr)
top2 = np.argsort(spectrum)[-2:]
print(sorted(float(freqs[i]) for i in top2))
print(len(spectrum), "бинов, шаг", round(float(freqs[1]), 3), "Гц")

#### ❓ **Вопрос**: Почему пики оказались ровно на 440.0 и 660.0, без дробной части?

<details>

<summary><strong>Ответ</strong></summary>

Шаг частотной сетки равен `sr / N`, то есть `1 / длительность` — из второй строки вывода видно, что он `0.5` Гц (запись длится 2 секунды). И 440, и 660 делятся на 0.5 нацело, поэтому каждая частота попала точно в свой бин.

Если бы тон был 440.3 Гц, энергия размазалась бы между соседними бинами и максимум встал бы на 440.5 — точнее шага сетки ответить нельзя. Улучшить разрешение по частоте можно только взяв более длинный кусок сигнала.

</details>

## 7. Зачем нужен STFT

Аккорд звучал неизменным все две секунды, поэтому спектр всей записи его описал.
Возьмём сигнал, который **меняется**: чирп — тон, чья частота линейно растёт с 200
до 2000 Гц за 4 секунды.

In [ ]:
tc = np.arange(sr * 4) / sr
phase = 2 * np.pi * (200 * tc + (2000 - 200) / (2 * 4.0) * tc**2)
chirp = 0.5 * np.sin(phase)
print(chirp.shape, round(len(chirp) / sr, 2), "секунды")

In [ ]:
sc = np.abs(np.fft.rfft(chirp))
fc = np.fft.rfftfreq(len(chirp), 1 / sr)
plt.plot(fc[:500], sc[:500])
plt.xlabel("частота, Гц")
plt.ylabel("амплитуда")
plt.show()

#### ❓ **Вопрос**: Спектр аккорда был двумя узкими пиками, а спектр чирпа — широкой полкой от 200 до 2000 Гц. Какой информации в этой полке не хватает?

<details>

<summary><strong>Ответ</strong></summary>

Не хватает **времени**. Полка честно говорит: «в записи встречались все частоты от 200 до 2000 Гц» — и это правда. Но точно такую же полку дала бы запись, где те же частоты звучали одновременно, или в обратном порядке, или вперемешку: преобразование Фурье усредняет по всей длительности.

А звук почти всегда меняется — речь, музыка, шум мотора. Отсюда идея следующего раздела: резать сигнал на короткие куски и считать спектр каждого отдельно.

</details>

## 8. STFT и спектрограмма

**STFT** (short-time Fourier transform) режет сигнал на перекрывающиеся **окна** и
считает спектр каждого. Параметров два:

- `n_fft` — длина окна в сэмплах: сколько сигнала видно за раз;
- `hop_length` — шаг между началами соседних окон.

Результат — комплексная матрица «частотные бины × кадры», она и называется
спектрограммой.

In [ ]:
S = librosa.stft(chirp, n_fft=2048, hop_length=512)
print(S.shape, S.dtype)
print(2048 // 2 + 1, 1 + len(chirp) // 512)

#### ❓ **Вопрос**: Откуда в форме `(1025, 173)` взялись именно 1025 и 173?

<details>

<summary><strong>Ответ</strong></summary>

1025 — это `n_fft // 2 + 1`: спектр вещественного сигнала симметричен, поэтому хранится половина плюс нулевая частота. Ровно та же арифметика была у `rfft` в разделе 6.

173 — это `1 + len(chirp) // hop_length = 1 + 88200 // 512`: окна ставятся через каждые 512 сэмплов. По умолчанию `librosa` работает с `center=True` и дополняет сигнал по краям, поэтому кадр находится и для самого начала записи. Обе величины напечатаны второй строкой и совпали с формой.

Отсюда компромисс: увеличили `n_fft` — лучше разрешение по частоте, но хуже по времени (окно длиннее); уменьшили `hop_length` — больше кадров и больше памяти.

</details>

Амплитуды в спектрограмме различаются на порядки, и на линейной шкале виден
только самый громкий бин. Поэтому переходят к **децибелам** — логарифмической
шкале. `amplitude_to_db` берёт логарифм модуля, а `ref=np.max` делает точкой
отсчёта максимум записи, так что 0 дБ — самое громкое место.

In [ ]:
S_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)
print(round(float(S_db.max()), 1), round(float(S_db.min()), 1))
print(S_db.shape, S_db.dtype)

Нижняя граница -80 дБ — это умолчание `top_db=80`: всё, что тише максимума в
10 000 раз по амплитуде, обрезается, чтобы тишина не превращалась в минус
бесконечность.

Теперь `S_db` — обычный двумерный массив вещественных чисел, то есть картинка,
такая же, как в семинаре 15. Рисовать её умеет `librosa.display.specshow`: он
подписывает оси временем и частотой, а не номерами кадров и бинов.

In [ ]:
import librosa.display

librosa.display.specshow(S_db, sr=sr, hop_length=512, x_axis="time", y_axis="hz")
plt.colorbar(format="%+2.0f dB")
plt.ylim(0, 3000)
plt.show()

Видна диагональ: частота растёт со временем. Проверим это численно — найдём
бин с максимумом энергии в первом и последнем кадре.

In [ ]:
bins = librosa.fft_frequencies(sr=sr, n_fft=2048)
first = int(np.argmax(np.abs(S)[:, 0]))
last = int(np.argmax(np.abs(S)[:, -1]))
print(first, round(float(bins[first]), 1))
print(last, round(float(bins[last]), 1))

#### ❓ **Вопрос**: Номера бинов — 19 и 185. Как из них получаются 204.6 и 1991.8 Гц и почему не ровно 200 и 2000?

<details>

<summary><strong>Ответ</strong></summary>

Бины STFT равномерны по частоте с шагом `sr / n_fft = 22050 / 2048 ≈ 10.77` Гц, поэтому частота бина — это его номер, умноженный на шаг: `19 · 10.77 ≈ 204.6`, `185 · 10.77 ≈ 1991.8`. Именно такую таблицу и возвращает `librosa.fft_frequencies`.

Ровно 200 и 2000 получиться не могло: этих значений на сетке нет, ближайшие бины отстоят меньше чем на полшага. Точнее одного бина STFT ответить не умеет — это та же история, что с шагом 0.5 Гц в разделе 6, только окно короче и шаг сетки грубее.

</details>

## 9. Мел-спектрограмма

1025 бинов — это много и неэффективно: человек различает 200 и 300 Гц легко, а 5000
и 5100 — почти никак. **Мел-шкала** — шкала частоты, растянутая внизу и сжатая
вверху, ближе к тому, как слышит ухо. Мел-спектрограмма собирает соседние бины STFT
в `n_mels` полос по этой шкале.

In [ ]:
mel = librosa.feature.melspectrogram(y=chirp, sr=sr, n_fft=2048,
                                     hop_length=512, n_mels=64)
mel_db = librosa.power_to_db(mel, ref=np.max)
print(mel.shape, mel_db.shape)

Кадров столько же — 173, а частотных строк стало 64 вместо 1025. Обратите
внимание на `power_to_db`, а не `amplitude_to_db`: `melspectrogram` возвращает
**мощность** (квадрат амплитуды), и множитель перед логарифмом у неё другой.

Посмотрим на саму шкалу: центры мел-полос неравномерны по герцам.

In [ ]:
mf = librosa.mel_frequencies(n_mels=64, fmax=sr / 2)
print(np.round(mf[:4], 1))
print(round(float(np.diff(mf)[0]), 1), round(float(np.diff(mf)[-1]), 1))

#### ❓ **Вопрос**: Шаг между соседними полосами внизу — 52.8 Гц, вверху — 584.4 Гц. Что это даёт признакам модели?

<details>

<summary><strong>Ответ</strong></summary>

Низ описан подробно, верх — грубо, одной полосой почти на 600 Гц. Для речи и музыки это ровно то, что нужно: основной тон голоса и первые форманты лежат внизу, а наверху информации мало и она размазана.

Выигрыш виден по формам из предыдущей ячейки: вместо матрицы `1025 × 173` модель получает `64 × 173` — в 16 раз меньше чисел при почти той же полезной информации. Поэтому входом почти всех аудиомоделей служит именно мел-спектрограмма в дБ, а не сырой звук и не полный STFT.

</details>

In [ ]:
librosa.display.specshow(mel_db, sr=sr, hop_length=512,
                         x_axis="time", y_axis="mel")
plt.colorbar(format="%+2.0f dB")
plt.show()

Та же диагональ, но ось частоты сжата сверху: на мел-шкале нижняя часть
занимает бо́льшую долю картинки. Это и есть «признаки, которые можно подать
модели», — и ровно это просят в домашке.

## 10. То же самое в torchaudio

Если дальше идёт обучение на PyTorch, признаки удобнее считать сразу тензорами:
они попадают на то же устройство, что и модель, и участвуют в `Dataset` без
конвертаций. `torchaudio.load` возвращает **пару** «тензор, частота», как
`sf.read`, но тензор.

Оговорка про установку: начиная с версии 2.9 декодирование в `torchaudio`
вынесено в отдельный пакет `torchcodec`. Без него `load` падает с `ImportError`,
поэтому ставить надо оба: `pip install torchaudio torchcodec`.

In [ ]:
import torch
import torchaudio

wav, sr_t = torchaudio.load(work / "stereo.wav")
print(type(wav).__name__, wav.shape, wav.dtype, sr_t)

#### ❓ **Вопрос**: У `soundfile` тот же файл читался как `(44100, 2)` типа `float64`, а здесь — `torch.Size([2, 44100])` типа `float32`. Что из этого важно помнить?

<details>

<summary><strong>Ответ</strong></summary>

Оба отличия. Форма — `[каналы, сэмплы]`: канал первым, то же соглашение, что у `librosa` с `mono=False` в разделе 4, и противоположное `soundfile`. Тип — `float32`, а не `float64`: модели PyTorch по умолчанию работают в одинарной точности, и при переносе массива из numpy нужен явный `.float()`, иначе слой упадёт на несовпадении типов.

А вот ресемплинга здесь не произошло: `torchaudio.load` отдал 22050 — частоту как в файле, в отличие от `librosa.load`. Приводить частоту, если надо, придётся самим (`torchaudio.transforms.Resample`).

</details>

Преобразования в `torchaudio` — это **модули** (`torch.nn.Module`), как слои
сети: объект создаётся один раз с параметрами, а потом вызывается на данных. Он
переносится на GPU вместе с моделью и работает сразу на батче.

In [ ]:
to_mel = torchaudio.transforms.MelSpectrogram(
    sample_rate=sr, n_fft=2048, hop_length=512, n_mels=64)
batch = torch.from_numpy(np.stack([chirp, chirp * 0.5])).float()
print(batch.shape, to_mel(batch).shape)

#### ❓ **Вопрос**: На вход слою пришёл тензор `[2, 88200]`, на выходе — `[2, 64, 173]`. Почему не понадобился цикл по записям?

<details>

<summary><strong>Ответ</strong></summary>

`MelSpectrogram` считает преобразование по **последнему** измерению, а все предыдущие трактует как батч. Поэтому `[2, 88200]` — это две записи по 88200 сэмплов, и к каждой приписалась матрица `64 × 173` с тем же смыслом, что дал `librosa` в разделе 9 (числа близки, но не равны — умолчания нормировки и окна у библиотек разные).

Обратная сторона: тензор `[2, 88200]` неотличим от одной стереозаписи, которую `torchaudio.load` вернул строкой выше. Что означает первое измерение, знает только автор кода.

</details>

## 11. Видео: кадры во времени

Видео — это последовательность кадров плюс частота кадров **fps** (frames per
second). Один кадр — обычная картинка из семинара 15; новое здесь только время:

`число кадров = fps · длительность`

Соберём учебный клип: белый квадрат едет вправо, 30 кадров при 10 кадрах в
секунду.

In [ ]:
frames = np.zeros((30, 48, 64, 3), dtype=np.uint8)
for i in range(30):
    frames[i, 20:28, 2 * i:2 * i + 8] = 255
print(frames.shape, frames.dtype)

Форма — `(кадры, высота, ширина, каналы)`: к знакомой картинке `H × W × C`
приписано измерение времени. Кодировать в mp4 будем внешним `ffmpeg`: подаём сырые
байты на стандартный вход, получаем файл.

In [ ]:
import subprocess

cmd = ("ffmpeg -y -loglevel error -f rawvideo -pix_fmt rgb24 "
       "-s 64x48 -r 10 -i - -c:v libx264 -pix_fmt yuv420p")
clip = work / "clip.mp4"
subprocess.run(cmd.split() + [str(clip)], input=frames.tobytes(), check=True)
print(clip.stat().st_size, "байт")

Метаданные читает `ffprobe` — он не декодирует видео, а смотрит заголовки,
поэтому работает мгновенно даже на большом файле. Это аналог `sf.info` для звука.

In [ ]:
probe = ("ffprobe -v error -select_streams v:0 -show_entries "
         "stream=width,height,r_frame_rate,nb_frames -of csv=p=0")
out = subprocess.run(probe.split() + [str(clip)], capture_output=True, text=True)
print(out.stdout.strip())

Обратный путь — декодировать в массив кадров. Самый переносимый способ тот же
`ffmpeg`: попросим его выдать сырые пиксели на стандартный выход и разберём
буфер.

In [ ]:
decode = f"ffmpeg -v error -i {clip} -f rawvideo -pix_fmt rgb24 -"
raw = subprocess.run(decode.split(), capture_output=True).stdout
back = np.frombuffer(raw, dtype=np.uint8).reshape(-1, 48, 64, 3)
print(back.shape, back.dtype)
print(int(np.abs(back.astype(int) - frames.astype(int)).max()))

#### ❓ **Вопрос**: Форма `back` — `(30, 48, 64, 3)`, а `ffprobe` напечатал `64,48,10/1,30`. Как получить длительность клипа и почему `reshape(-1, 48, 64, 3)` вообще сработал?

<details>

<summary><strong>Ответ</strong></summary>

Длительность — `30 / 10 = 3.0` секунды: число кадров, делённое на fps (`nb_frames = 30`, `r_frame_rate = 10/1`). Ровно та же формула, что для звука в разделе 1, только вместо сэмплов кадры.

`reshape` сработал потому, что размер кадра мы знали заранее: `ffmpeg` отдаёт сплошной поток байт без всяких разделителей, и разложить его на кадры можно, только зная высоту, ширину и формат пикселя (`rgb24` — 3 байта на пиксель). Возьмёте не те числа — `reshape` либо упадёт, либо тихо соберёт кашу.

</details>

Расхождение с исходными кадрами нулевое: наш синтетический клип из плоских
чёрных и белых прямоугольников кодек H.264 передал точно. В общем случае так не
бывает — H.264 сжимает **с потерями** и вдобавок хранит цветность в `yuv420p` с
вдвое меньшим разрешением, так что реальная съёмка после кодирования по пикселям
не совпадёт. Нужны кадры бит-в-бит — храните их без потерь (`-c:v ffv1` или
последовательность png).

Про `torchvision`: в свежих версиях (0.28) декодирование видео из него **убрано** —
`torchvision.io` остался только про изображения (`decode_image`, `read_image`).
Тензор кадров теперь отдаёт тот же `torchcodec`, что нужен и `torchaudio`.

In [ ]:
from torchcodec.decoders import VideoDecoder

decoder = VideoDecoder(str(clip))
print(len(decoder), decoder.metadata.average_fps)
print(decoder[:].shape, decoder[10].shape)

#### ❓ **Вопрос**: `ffmpeg` дал форму `(30, 48, 64, 3)`, а `VideoDecoder` — `[30, 3, 48, 64]`. Что изменилось и почему так сделано?

<details>

<summary><strong>Ответ</strong></summary>

Переставлены измерения: `(T, H, W, C)` против `[T, C, H, W]` — канал уехал с последнего места на второе. Это стандартная раскладка входа свёрточных слоёв PyTorch (`NCHW`), поэтому тензор из `VideoDecoder` можно подавать в модель без `permute`.

Заодно обратите внимание: у звука `torchaudio` тоже кладёт канал перед временем (`[каналы, сэмплы]`, раздел 10), а `ffmpeg` и `soundfile` — после. Одно и то же правило: перед использованием печатаем `.shape`.

</details>

## Итог

- Звук в памяти — массив чисел от -1 до 1; смысл ему даёт частота дискретизации:
  `длительность = число сэмплов / sr`.
- `soundfile`: `sf.read` → `(массив, sr)`, форма стерео `(сэмплы, каналы)`,
  `sf.info` читает метаданные без декодирования.
- `librosa.load` по умолчанию ресемплирует к 22050 Гц и сводит в моно — пишите
  `sr=None`, если этого не хотите; форма стерео у него `(каналы, сэмплы)`.
- Громкость правят делением на пик, а не умножением: за пределами -1..1 начинается
  клиппинг, а вместе с ним лишние гармоники.
- Спектр всей записи описывает только неизменный сигнал. Для меняющегося нужен
  **STFT**: форма `(n_fft // 2 + 1, 1 + N // hop_length)`, шаг по частоте
  `sr / n_fft`.
- Шкала дБ (`amplitude_to_db` для амплитуд, `power_to_db` для мощности) делает
  спектрограмму читаемой; мел-шкала сжимает 1025 бинов до `n_mels` полос — это и
  есть типичный вход аудиомодели.
- `torchaudio`: `load` → `(тензор [каналы, сэмплы] float32, sr)`, `transforms` —
  модули, работающие на батче и на GPU. Нужен пакет `torchcodec`.
- Видео — кадры плюс fps; метаданные быстрее всего смотреть через `ffprobe`,
  кадры разбирать через `ffmpeg` в `(T, H, W, C)` или `VideoDecoder` в
  `[T, C, H, W]`.

Домашка — спектрограмма аудиозаписи. Задачи семинара — в `tasks.md`, учебные
данные готовит `python3 assets/make_data.py`: бинарных файлов в репозитории нет,
всё генерируется детерминированно, поэтому числа в тестовых примерах у всех
одинаковые.